# Z5008 Big Data Lab — Lab 2: Delta Lake Foundations
**IIT Madras Zanzibar | Dr. Innocent Nyalala | Monday 16 March 2026**

---

## What we will do today
| # | Task | Time |
|---|------|------|
| 1 | SparkSession connected to MinIO + Delta Lake | 10 min |
| 2 | Write 10,000-row dataset to Delta Lake (Bronze) | 15 min |
| 3 | SQL queries: aggregations, fraud analysis | 10 min |
| 4 | Time travel: read past versions, RESTORE | 10 min |
| 5 | MERGE / UPSERT: apply CDC updates | 15 min |
| 6 | Schema enforcement and evolution | 5 min |
| 7 | OPTIMIZE + Z-ORDER + VACUUM | 5 min |

**Run each cell with Shift+Enter. Read all markdown cells — they explain what is happening.**

---
## Part 1: SparkSession — Connect Spark to MinIO + Delta Lake

This configuration does three things:
1. **Delta Lake extensions**: loads the Delta SQL parser and the `DeltaCatalog` so `delta.\`path\`` syntax works in SQL
2. **S3A connector**: tells Spark where MinIO lives (`http://minio:9000`) and how to authenticate
3. **Performance settings**: enables Adaptive Query Execution (AQE) and reduces shuffle partitions for our small local cluster

> **Note:** `minio:9000` works inside Docker's network. From your laptop browser, use `localhost:9001` for the console.

In [1]:
# 1. Force Jupyter to install the Delta library and its Java files locally
!pip install delta-spark==3.1.0

from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

# 2. Build the session WITHOUT the spark.jars.packages line
builder = (
    SparkSession.builder
    .appName("Z5008-Lab2-DeltaLake")
    # ── Delta Lake extensions ──────────────────────────────────────────────
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    # ── S3A connector: talk to MinIO ───────────────────────────────────────
    .config("spark.hadoop.fs.s3a.endpoint",            "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key",          "admin")
    .config("spark.hadoop.fs.s3a.secret.key",          "bigdata123")
    .config("spark.hadoop.fs.s3a.path.style.access",   "true")
    .config("spark.hadoop.fs.s3a.impl",                "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    # ── Performance ────────────────────────────────────────────────────────
    .config("spark.sql.adaptive.enabled",       "true")
    .config("spark.sql.shuffle.partitions",     "8")
)

# 3. Let Delta automatically inject its own JARs, and tell it to fetch the AWS JARs we need
spark = configure_spark_with_delta_pip(
    builder, 
    extra_packages=["org.apache.hadoop:hadoop-aws:3.3.4", "com.amazonaws:aws-java-sdk-bundle:1.12.262"]
).getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f" SparkSession ready! Spark version: {spark.version}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 42.2 kB/s eta 0:00:00a 0:00:01


PySparkRuntimeError: [JAVA_GATEWAY_EXITED] Java gateway process exited before sending its port number.

---
## Part 2: Create the Dataset and Write to Delta Lake (Bronze)

We generate 10,000 synthetic East African retail transactions with 9 columns.

**Writing to Delta Lake:**
- `format("delta")` — use Delta instead of plain Parquet
- `mode("overwrite")` — replace the table if it already exists (creates version 0)
- `partitionBy("city")` — creates separate sub-folders per city. Queries filtering by city will only read that city's files (**partition pruning**)

**After writing:** Go to http://localhost:9001 → warehouse → bronze → transactions  
You will see: `_delta_log/` folder + `city=Nairobi/`, `city=Zanzibar/`, etc.

In [ ]:
import pandas as pd
import random
from pyspark.sql.types import (
    StructType, StructField, IntegerType, StringType, 
    DoubleType, TimestampType, BooleanType
)

# 1. Generate the synthetic data using Pandas
random.seed(42)
n = 10000

pdf = pd.DataFrame({
    "txn_id":      range(1, n + 1),
    "customer_id": [f"C{i % 500 + 1:04d}" for i in range(n)],
    "merchant":    ["ShopRite", "Nakumatt", "Jumia", "M-Pesa", "KCB Bank"] * 2000,
    "city":        ["Nairobi", "Dar es Salaam", "Zanzibar", "Kampala", "Mombasa"] * 2000,
    "category":    ["groceries", "electronics", "finance", "transport", "dining"] * 2000,
    "amount_usd":  [round(abs(random.gauss(150, 120)), 2) for _ in range(n)],
    "currency":    ["KES", "TZS", "TZS", "KES", "KES"] * 2000,
    "txn_date":    pd.date_range("2026-01-01", periods=n, freq="h"),
    "is_fraud":    [False] * 9800 + [True] * 200
})

# 2. Define an explicit Spark Schema (Production Best Practice)
txn_schema = StructType([
    StructField("txn_id", IntegerType(), False),
    StructField("customer_id", StringType(), True),
    StructField("merchant", StringType(), True),
    StructField("city", StringType(), True),
    StructField("category", StringType(), True),
    StructField("amount_usd", DoubleType(), True),
    StructField("currency", StringType(), True),
    StructField("txn_date", TimestampType(), True),
    StructField("is_fraud", BooleanType(), True)
])

# 3. Create the Spark DataFrame safely
df = spark.createDataFrame(pdf, schema=txn_schema)

# 4. Verify the results
print(f"Created DataFrame: {df.count()} rows")
df.printSchema()
df.show(5, truncate=False)


In [ ]:
TABLE = "s3a://warehouse/bronze/transactions"

(
    df.write
    .format("delta")
    .mode("overwrite")
    .partitionBy("city")
    .save(TABLE)
)

print("Written to Delta Lake!")
print(f"Path: {TABLE}")
print()
print("Now open: http://localhost:9001")
print("Browse: warehouse -> bronze -> transactions")
print("You will see the _delta_log/ folder and city=.../ partitions.")

In [ ]:
# Read back and confirm
df_delta = spark.read.format("delta").load(TABLE)
print(f"Rows read back from Delta: {df_delta.count()}")

# Register as SQL view
df_delta.createOrReplaceTempView("transactions")
print("Registered as SQL view: 'transactions'")

---
## Part 3: SQL Queries

Notice: because the table is partitioned by `city`, a query filtering `WHERE city = 'Nairobi'` only reads the `city=Nairobi/` folder — it physically skips all other files. This is **partition pruning** and is visible in the Spark UI physical plan.

In [ ]:
# Query 1: Total spend and fraud rate by city
print("=== Spend and Fraud Rate by City ===")
spark.sql("""
    SELECT city,
           COUNT(*)                                             AS num_txns,
           ROUND(SUM(amount_usd), 2)                           AS total_usd,
           ROUND(AVG(amount_usd), 2)                           AS avg_usd,
           SUM(CAST(is_fraud AS INT))                          AS fraud_count,
           ROUND(100.0 * SUM(CAST(is_fraud AS INT)) / COUNT(*), 2) AS fraud_pct
    FROM transactions
    GROUP BY city
    ORDER BY total_usd DESC
""").show()

In [ ]:
# Query 2: Top merchants by fraud rate (for transactions > $50)
print("=== Merchant Fraud Analysis (amount > $50) ===")
spark.sql("""
    SELECT merchant,
           COUNT(*)                                     AS total_txns,
           SUM(CAST(is_fraud AS INT))                  AS fraud_count,
           ROUND(100.0 * AVG(CAST(is_fraud AS INT)), 2) AS fraud_rate_pct,
           ROUND(AVG(amount_usd), 2)                   AS avg_amount_usd
    FROM transactions
    WHERE amount_usd > 50
    GROUP BY merchant
    ORDER BY fraud_rate_pct DESC
""").show()

In [ ]:
# Query 3: Top 10 customers by total spend (the "VIP" customers)
print("=== Top 10 Customers by Lifetime Spend ===")
spark.sql("""
    SELECT customer_id,
           COUNT(*)                   AS txn_count,
           ROUND(SUM(amount_usd), 2)  AS total_spend_usd,
           ROUND(MAX(amount_usd), 2)  AS max_single_txn,
           MAX(txn_date)              AS last_seen
    FROM transactions
    GROUP BY customer_id
    ORDER BY total_spend_usd DESC
    LIMIT 10
""").show()

In [ ]:
# Query 4: Using PySpark DataFrame API (equivalent to SQL, but in Python)
from pyspark.sql import functions as F
from pyspark.sql.window import Window

print("=== Daily transaction volume (first 10 days) ===")
(
    df_delta
    .withColumn("date", F.to_date("txn_date"))
    .groupBy("date")
    .agg(
        F.count("*").alias("num_txns"),
        F.round(F.sum("amount_usd"), 2).alias("daily_total"),
        F.sum(F.col("is_fraud").cast("int")).alias("fraud_count")
    )
    .orderBy("date")
    .show(10)
)

---
## Part 4: Time Travel

**How time travel works in Delta Lake:**
- Every write creates a new version in `_delta_log/00000000000000000001.json`, etc.
- Reading version N = Delta loads only the log entries for version N, which point to the Parquet files that were "current" at that version
- Old Parquet files are physically kept on disk until `VACUUM` removes them
- `RESTORE` atomically rewrites the table to a past state (adds a new commit entry pointing to the old files)

In [ ]:
# Create version 1: append 500 new Zanzibar Equity Bank transactions
pdf2 = pd.DataFrame({
    "txn_id":      range(10001, 10501),
    "customer_id": [f"C{i % 500 + 1:04d}" for i in range(500)],
    "merchant":    ["Equity Bank"] * 500,
    "city":        ["Zanzibar"] * 500,
    "category":    ["finance"] * 500,
    "amount_usd":  [500.0] * 500,
    "currency":    ["TZS"] * 500,
    "txn_date":    pd.date_range("2026-03-16", periods=500, freq="min"),
    "is_fraud":    [False] * 500,
})

# 2. Fix the DataType Error: Force txn_id to be an Integer
df_append = spark.createDataFrame(pdf2).withColumn("txn_id", F.col("txn_id").cast("integer"))

# 3. Append the data to create Version 1
df_append.write.format("delta").mode("append").save(TABLE)
print("Appended 500 rows (this is now version 1)")

# ---------------------------------------------------------
# TIME TRAVEL COMMANDS
# ---------------------------------------------------------
print("\n--- Delta Lake History ---")
spark.sql(f"DESCRIBE HISTORY delta.`{TABLE}`").select("version", "timestamp", "operation").show(truncate=False)

# Read Version 0 (Original 10,000 rows)
v0_count = spark.read.format("delta").option("versionAsOf", 0).load(TABLE).count()
print(f"Row count in Version 0: {v0_count}")

# Read Version 1 (New 10,500 rows)
v1_count = spark.read.format("delta").option("versionAsOf", 1).load(TABLE).count()
print(f"Row count in Version 1: {v1_count}")

In [ ]:
# See the full version history
print("=== Delta Lake Transaction Log History ===")
spark.sql(f"""
    DESCRIBE HISTORY delta.`{TABLE}`
""").select("version", "timestamp", "operation", "operationMetrics") \
    .show(10, truncate=False)

In [ ]:
# Read version 0 (before the append)
v0 = spark.read.format("delta").option("versionAsOf", 0).load(TABLE)
v1 = spark.read.format("delta").option("versionAsOf", 1).load(TABLE)

print(f"Version 0 row count: {v0.count():,}  (should be 10,000)")
print(f"Version 1 row count: {v1.count():,}  (should be 10,500)")

# Compare Zanzibar in each version
z0 = v0.filter("city = 'Zanzibar'").count()
z1 = v1.filter("city = 'Zanzibar'").count()
print(f"\nZanzibar rows in version 0: {z0:,}")
print(f"Zanzibar rows in version 1: {z1:,}")
print(f"Delta (should be 500): {z1 - z0}")

In [ ]:
# RESTORE: undo the append — go back to version 0
print("Restoring to version 0 (undoing the append)...")
spark.sql(f"RESTORE TABLE delta.`{TABLE}` TO VERSION AS OF 0")

after_restore = spark.read.format("delta").load(TABLE)
print(f"After RESTORE: {after_restore.count():,} rows  (should be 10,000)")

# The history still shows all versions — RESTORE adds a new commit
print("\nHistory now shows a RESTORE entry:")
spark.sql(f"""
    DESCRIBE HISTORY delta.`{TABLE}`
""").select("version", "operation").show(5)

---
## Part 5: MERGE / UPSERT

**Use case:** You have a Silver customers table. Every hour, your CDC pipeline sends a batch of:
- Updated customer records (city changed, account updated)
- Deleted customers (cancelled accounts)
- New customers (just signed up)

MERGE applies all three in one atomic transaction — no full table rewrite needed.

In [ ]:
from delta.tables import DeltaTable

CUST = "s3a://warehouse/silver/customers"

# Build Silver customers: one row per customer with aggregated stats
(
    spark.sql(f"""
        SELECT customer_id,
               FIRST(city)               AS city,
               COUNT(*)                  AS txn_count,
               ROUND(SUM(amount_usd), 2) AS lifetime_value_usd,
               MAX(txn_date)             AS last_txn_date
        FROM delta.`{TABLE}`
        GROUP BY customer_id
    """)
    .write.format("delta").mode("overwrite").save(CUST)
)

print(f"Silver customers: {spark.read.format('delta').load(CUST).count()} rows")
spark.read.format("delta").load(CUST).show(5)

In [ ]:
# Incoming CDC batch: 2 updates + 1 delete + 1 new insert
updates = spark.createDataFrame([
    ("C0001", "Arusha",  9999.99, "active"),     # UPDATE: city changed
    ("C0002", "Mombasa", 1234.56, "active"),     # UPDATE: lifetime value changed
    ("C0003", None,      0.0,     "cancelled"),  # DELETE: account closed
    ("C0999", "Zanzibar",0.0,     "new"),        # INSERT: brand new customer
], ["customer_id", "city", "lifetime_value_usd", "status"])

print("Incoming updates:")
updates.show()

# Run the MERGE
target = DeltaTable.forPath(spark, CUST)

(
    target.alias("tgt")
    .merge(updates.alias("src"), "tgt.customer_id = src.customer_id")
    .whenMatchedDelete(condition="src.status = 'cancelled'")
    .whenMatchedUpdate(set={
        "city":               "src.city",
        "lifetime_value_usd": "src.lifetime_value_usd"
    })
    .whenNotMatchedInsert(values={
        "customer_id":        "src.customer_id",
        "city":               "src.city",
        "txn_count":          "0",
        "lifetime_value_usd": "src.lifetime_value_usd",
        "last_txn_date":      "current_timestamp()"
    })
    .execute()
)
print("MERGE executed!")

In [ ]:
# Verify the MERGE results
result = spark.read.format("delta").load(CUST)

print("=== Check C0001 (should be Arusha now) ===")
result.filter("customer_id = 'C0001'").show()

print("=== Check C0003 (should be GONE — deleted) ===")
c0003 = result.filter("customer_id = 'C0003'").count()
print(f"C0003 rows: {c0003}  (expected: 0)")

print("=== Check C0999 (should be NEW — inserted) ===")
result.filter("customer_id = 'C0999'").show()

# MERGE history
print("=== MERGE recorded in Delta history ===")
spark.sql(f"DESCRIBE HISTORY delta.`{CUST}`") \
    .select("version", "operation", "operationMetrics").show(3, truncate=False)

---
## Part 6: Schema Enforcement and Evolution

Delta Lake's schema enforcement is **on by default** — it protects your table from accidentally ingesting data with the wrong shape.  
Schema evolution is **opt-in** — you use `mergeSchema=true` only when you deliberately want to add new columns.

In [ ]:
# Test schema enforcement: try to write an extra column that doesn't exist
bad_df = spark.createDataFrame(
    [(99999, "C0001", "Nakumatt", "Nairobi", "groceries",
      100.0, "KES", "2026-03-16", False, "THIS COLUMN DOES NOT EXIST")],
    ["txn_id","customer_id","merchant","city","category",
     "amount_usd","currency","txn_date","is_fraud","unknown_column"]
)

try:
    bad_df.write.format("delta").mode("append").save(TABLE)
    print("ERROR: This should have been rejected!")
except Exception as e:
    print("GOOD — Delta Lake REJECTED the write as expected.")
    print(f"Error type: {type(e).__name__}")
    print(f"Message: {str(e)[:300]}")

In [ ]:
# Schema evolution: deliberately add a new 'discount' column
# (simulates: new app version now sends a discount field)
import pyspark.sql.functions as F

# 1. Create the DataFrame
new_df = spark.createDataFrame(
    [(99999, "C0001", "Nakumatt", "Nairobi", "groceries",
      100.0, "KES", "2026-03-16 00:00:00", False, 0.10)],
    ["txn_id","customer_id","merchant","city","category",
     "amount_usd","currency","txn_date","is_fraud","discount"]
)

# 2. Fix the data types to match your Bronze table
new_df_fixed = (
    new_df
    .withColumn("txn_id", F.col("txn_id").cast("integer"))
    .withColumn("txn_date", F.col("txn_date").cast("timestamp"))
)

# 3. Write with Schema Evolution enabled
(
    new_df_fixed.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")   # opt-in: allow new columns
    .save(TABLE)
)

print("✅ Schema evolved! New column 'discount' added.")
print("\nNew schema:")
spark.read.format("delta").load(TABLE).printSchema()

# Old rows will have discount = null
print("\nSample — old rows have discount = null:")
spark.read.format("delta").load(TABLE) \
    .select("txn_id", "amount_usd", "discount") \
    .orderBy(F.col("discount").desc_nulls_last()) \
    .show(5)

---
## Part 7: OPTIMIZE + Z-ORDER + VACUUM

**OPTIMIZE** compacts many small Parquet files into larger ones (~1GB each).  
**Z-ORDER** reorganises which rows go in which file to co-locate by column values — queries filtering by `customer_id` or `txn_date` will skip far more files.

After running OPTIMIZE, check MinIO console again — the file count and sizes will have changed.

In [ ]:
# Inspect table details before OPTIMIZE
print("=== Table details BEFORE OPTIMIZE ===")
spark.sql(f"DESCRIBE DETAIL delta.`{TABLE}`") \
    .select("numFiles", "sizeInBytes", "partitionColumns") \
    .show(truncate=False)

In [ ]:
# OPTIMIZE + Z-ORDER: compact and co-locate by customer and date
print("Running OPTIMIZE with Z-ORDER (this may take 30-60 seconds)...")
result = spark.sql(f"""
    OPTIMIZE delta.`{TABLE}`
    ZORDER BY (customer_id, txn_date)
""")
result.show(truncate=False)
print("OPTIMIZE complete. Check MinIO console — fewer, larger files.")

In [ ]:
# Inspect table details after OPTIMIZE
print("=== Table details AFTER OPTIMIZE ===")
spark.sql(f"DESCRIBE DETAIL delta.`{TABLE}`") \
    .select("numFiles", "sizeInBytes", "partitionColumns") \
    .show(truncate=False)

In [ ]:
# VACUUM dry run: see what would be deleted
print("=== VACUUM DRY RUN (nothing deleted yet) ===")
spark.sql(f"""
    VACUUM delta.`{TABLE}` DRY RUN
""").show(10, truncate=False)

# NOTE: We will NOT run the actual VACUUM in the lab
# because we want to keep time travel available for the assignment.
# To actually run: spark.sql(f"VACUUM delta.`{TABLE}` RETAIN 24 HOURS")

---
## Part 8: Bonus — Window Functions on Delta Table

Window functions allow you to compute running totals, rankings, and moving averages without GROUP BY.
These are very common in financial data analysis.

In [ ]:
from pyspark.sql.window import Window

# Rank customers within each city by total spend
df_ranked = (
    spark.sql(f"""
        SELECT customer_id, city,
               ROUND(SUM(amount_usd), 2) AS total_spend
        FROM delta.`{TABLE}`
        GROUP BY customer_id, city
    """)
)

window_spec = Window.partitionBy("city").orderBy(F.desc("total_spend"))

result = (
    df_ranked
    .withColumn("rank_in_city", F.rank().over(window_spec))
    .withColumn("running_total",
                F.round(F.sum("total_spend").over(
                    window_spec.rowsBetween(Window.unboundedPreceding, 0)), 2))
    .filter("rank_in_city <= 3")   # top 3 spenders per city
    .orderBy("city", "rank_in_city")
)

print("=== Top 3 Spenders Per City (with running total) ===")
result.show(20)

---
## Summary

### Tables created today
| Table | Path | Rows | Format |
|-------|------|------|--------|
| Bronze transactions | `s3a://warehouse/bronze/transactions` | 10,001 (after schema evolution row) | Delta, partitioned by city |
| Silver customers | `s3a://warehouse/silver/customers` | 500 customers | Delta |

### Key commands reference
```python
# Write
df.write.format("delta").mode("overwrite").partitionBy("col").save(PATH)

# Read
spark.read.format("delta").load(PATH)

# Time travel
spark.read.format("delta").option("versionAsOf", N).load(PATH)

# History
spark.sql(f"DESCRIBE HISTORY delta.`{PATH}`").show()

# Restore
spark.sql(f"RESTORE TABLE delta.`{PATH}` TO VERSION AS OF N")

# MERGE
DeltaTable.forPath(spark, PATH).alias("t").merge(updates, "t.id = s.id")
    .whenMatchedUpdate(...).whenNotMatchedInsert(...).execute()

# OPTIMIZE
spark.sql(f"OPTIMIZE delta.`{PATH}` ZORDER BY (col1, col2)")

# VACUUM
spark.sql(f"VACUUM delta.`{PATH}` RETAIN 72 HOURS")
```

**Next week (Lab 3, 23 March):** Delta Lake Advanced (Change Data Feed, streaming writes) + Apache Iceberg.

**Assignment 2 due: Sunday 22 March, 23:59 on Moodle.**  
See `week02_assignment.pdf` for details.